# 🤖 Notebook 03 — Predictive Modeling & Scenario Projections
## Issaquah Creek Salmon Return Study | Summer 2025

This notebook:
1. Train baseline linear regression model
2. Train XGBoost model with time-series cross-validation
3. Evaluate both models on held-out 2011–2025 test data
4. Feature importance — what drives Issaquah Creek returns most?
5. Three-scenario projections through 2040
6. Community-ready summary chart

In [ ]:
import sys; sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

from src.features import build_features, get_model_features
from src.model import (
    temporal_split,
    train_linear_baseline,
    train_xgboost,
    plot_feature_importance,
    plot_actual_vs_predicted,
    run_scenario_projections,
    plot_scenario_projections,
    SCENARIOS,
)

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
FIG_DIR = Path('../outputs/figures')

df = build_features(pd.read_csv('../data/processed/issaquah_creek_master.csv'))
print(f'Dataset ready: {df.shape}')

## 1. Prepare Model Features

In [ ]:
TARGET = 'chinook_total'   # Change to 'coho_total' for Coho model

# Add water_year to X so temporal_split can use it
X, y, feature_names = get_model_features(df, target=TARGET)
X['water_year'] = df.loc[X.index, 'water_year'].values

print(f'Target: {TARGET}')
print(f'Features: {len(feature_names)}')
print(f'Samples: {len(X)}')
print(f'Year range: {X["water_year"].min()}–{X["water_year"].max()}')

In [ ]:
# Temporal train/test split: train 1985–2010, test 2011–present
X_train, X_test, y_train, y_test, years_train, years_test = temporal_split(
    X, y, train_end_year=2010
)

print(f'Training set: {len(X_train)} years ({years_train.min()}–{years_train.max()})')
print(f'Test set:     {len(X_test)} years ({years_test.min()}–{years_test.max()})')

## 2. Baseline Linear Model

In [ ]:
print('Training Ridge Regression baseline...')
linear_results = train_linear_baseline(X_train, y_train, X_test, y_test)

print('\nTop 10 features by coefficient magnitude (standardized):')
print(linear_results['coefficients'].head(10).to_string(index=False))

## 3. XGBoost Model

In [ ]:
print('Training XGBoost model with time-series CV...')
xgb_results = train_xgboost(X_train, y_train, X_test, y_test)

print('\nTop 15 features by XGBoost importance:')
print(xgb_results['importance'].head(15).to_string(index=False))

## 4. Model Comparison

In [ ]:
print('\n' + '='*50)
print('MODEL PERFORMANCE COMPARISON — Test Set (2011–2025)')
print('='*50)
print(f'{"Metric":<12} {"Linear Baseline":>18} {"XGBoost":>12}')
print('-'*50)
for metric in ['r2', 'mae', 'rmse']:
    lin = linear_results['metrics'][metric]
    xgb_val = xgb_results['metrics'][metric]
    fmt = '.3f' if metric == 'r2' else ',.0f'
    print(f'{metric.upper():<12} {lin:>{18}{fmt}} {xgb_val:>{12}{fmt}}')

best_model = 'XGBoost' if xgb_results['metrics']['r2'] > linear_results['metrics']['r2'] else 'Linear'
print(f'\n✓ Best model: {best_model}')
print(f'  Target R² threshold (≥ 0.65): {"PASSED" if xgb_results["metrics"]["r2"] >= 0.65 else "NOT YET — review features"}')

In [ ]:
# Actual vs. Predicted plot
train_preds = xgb_results['model'].predict(X_train)

plot_actual_vs_predicted(
    years_train, y_train, train_preds,
    years_test, y_test, xgb_results['predictions'],
    species='Chinook',
    filename='07_actual_vs_predicted_chinook.png'
)

## 5. Feature Importance — What Drives Issaquah Creek Returns?

In [ ]:
plot_feature_importance(
    xgb_results['importance'],
    top_n=15,
    title='What Most Affects Issaquah Creek Chinook Returns?\nXGBoost Feature Importance',
    filename='08_feature_importance_chinook.png'
)

## 6. Scenario Projections — Issaquah Creek Through 2040

In [ ]:
# Print scenario descriptions
print('SCENARIO DEFINITIONS')
print('='*65)
for name, sc in SCENARIOS.items():
    print(f'\n{name.upper()}: {sc["description"]}')
    for feat, delta in sc['adjustments'].items():
        print(f'  {feat}: {delta:+.1f} per year')

In [ ]:
# Use last available data row as starting point for projections
model_feature_cols = [c for c in X_train.columns if c != 'water_year']
last_row = X_test.drop(columns=['water_year']).iloc[-1]

projection_years = list(range(2026, 2041))

print(f'Projecting from {X_test["water_year"].max()} baseline through 2040...')
projections = run_scenario_projections(
    xgb_results['model'],
    last_known_row=last_row,
    model_features=model_feature_cols,
    projection_years=projection_years,
    species='Chinook'
)

In [ ]:
# Scenario projection chart
hist_df = df[['water_year','chinook_total']].dropna()

plot_scenario_projections(
    historical_years=hist_df['water_year'],
    historical_returns=hist_df['chinook_total'],
    projections=projections,
    species='Chinook',
    filename='09_scenario_projections_chinook.png'
)

In [ ]:
# Summary table for awareness report
print('SCENARIO PROJECTION SUMMARY — 2040 Outlook')
print('=' * 60)
print(f'2025 actual Chinook returns: 4,955')
print()
print(f'{"Scenario":<35} {"2030 Proj":>12} {"2040 Proj":>12} {"vs 2025":>10}')
print('-' * 60)
for name, series in projections.items():
    proj_2030 = series.get(2030, np.nan)
    proj_2040 = series.get(2040, np.nan)
    pct_change = (proj_2040 - 4955) / 4955 * 100
    sc_label = SCENARIOS[name]['label'].replace('\n', ' ')
    print(f'{sc_label:<35} {proj_2030:>12,.0f} {proj_2040:>12,.0f} {pct_change:>+10.1f}%')

## 7. Modeling Summary

*(Fill in after running on real data)*

| Model | Test R² | MAE | RMSE |
|-------|---------|-----|------|
| Linear Baseline | TBD | TBD | TBD |
| XGBoost | TBD | TBD | TBD |

**Top 3 features by XGBoost importance:**
1. TBD
2. TBD
3. TBD

**2040 Scenario Projections:**
- Optimistic: TBD returns
- Baseline: TBD returns
- Pessimistic: TBD returns

**➡ Proceed to `outputs/report/` for the Awareness Report draft.**